# Account.py

In [107]:
import numpy as np


class AssetAccount(dict):
    """
    하나에 자산군에 대한 정보를 저장한다.(ex) 주식, 금, 외환 등)
    """

    def __init__(self, exchange, cost=(0.3, 0.03, 0), 출력=True):
        self._출력 = 출력
        self.type = "Account"
        self._exchange = exchange

        self._date = None
        self._asset_info = None  # (날짜, 이름, 데이터) 3개의 axis로 구성된 array

        # self._balance = dict()
        self._tax, self._fee, self._slippage = cost

    def buy(self, names, 주문가격, 주문수량):
        pass

    def sell(self, names, 주문가격, 주문수량):
        pass

    def _add_assets(self, names, 주문가격, 현재가, 주문수량):
        for i in range(len(names)):
            name = names[i]
            수량 = 주문수량[i]
            가격 = 주문가격[i]

            if name not in self.keys():
                self[name] = {"현재가": 현재가[i], "평단가": 가격, "보유수량": 수량}
            else:
                self[name]["평단가"] = (수량 * 가격
                                              + self[name]["평단가"]
                                              * self[name]["보유수량"]) / (수량 + self[name]["보유수량"])
                self[name]["보유수량"] += 수량

    def _remove_assets(self, names, 주문가격, 주문수량):
        for i in range(len(names)):
            self[names[i]]["보유수량"] -= 주문수량[i]

            if self[names[i]]["보유수량"] == 0:
                del self[names[i]]

    def get_total_balance(self):
        total_balance = 0
        for name in self.keys():
            total_balance += self[name]["현재가"] * self[name]["보유수량"]

        return total_balance

    def _get_current_price(self):
        names = list(self.keys())
        current_price = self._exchange.get_assets_info(codes=names, fields=["현재가"])

        return names, current_price

    def _apply_current_price(self):
        names, current_price = self._get_current_price()
        for i in range(len(names)):
            self[names[i]]["현재가"] = current_price[i]

    def update_from_agent(self, date):
        self._date = date
        self._apply_current_price()

    def reset_from_agent(self, date):
        self._date = date
        for key in list(self.keys()):
            del self[key]


class StockAccount(AssetAccount):
    def __init__(self, exchange, cost=(0.3, 0.03, 0), 출력=True):
        AssetAccount.__init__(self, exchange, cost=cost, 출력=출력)

    def buy(self, names, 주문가격, 주문수량, 주문종류=None, 주문시간=None):
        체결가, 현재가 = self._exchange.buy(names, 주문가격, 주문종류=주문종류, 주문시간=주문시간)
        
        체결 = ~np.isnan(체결가)
        if self._출력:
            for i in range(len(체결)):
                if 체결[i]:
                    print("매수 체결 : ", names[i], "체결가 : ", 체결가[i], "주문수량 : ", 주문수량[i])
                else:
                    print("매수 실패 : ", names[i], "주문가", 주문가격[i], "주문수량 : ", 주문수량[i])

        names = np.array(names)[체결]
        체결가, 현재가 = 체결가[체결], 현재가[체결]
        주문수량 = np.array(주문수량)[체결]

        self._add_assets(names, 체결가, 현재가, 주문수량)

        거래대금 = np.sum(체결가 * 주문수량)
        return 거래대금

    def sell(self, names, 주문가격, 주문수량, 주문종류=None, 주문시간=None):
        체결가 = self._exchange.sell(names, 주문가격, 주문종류=주문종류, 주문시간=주문시간)

        체결 = ~np.isnan(체결가)
        if self._출력 == True:
            for i in range(len(체결)):
                if 체결[i]:
                    수익률 = (체결가[i] * (100 - self._tax - self._fee - self._slippage) / 100 - self[names[i]]["평단가"]) / \
                          self[names[i]]["평단가"] * 100
                    수익금 = (체결가[i] * (100 - self._tax - self._fee - self._slippage) / 100 - self[names[i]]["평단가"]) * \
                          self[names[i]]["보유수량"]

                    print("매도 체결 : ", names[i], "체결가 : ", 체결가[i], "주문수량 : ", 주문수량[i], 
                          " / 수익률(%) : ", 수익률, ", 수익금(원) : ", 수익금)
                else:
                    print("매도 실패 : ", names[i], "주문가 : ", 주문가격[i], "주문수량 : ", 주문수량[i])

                
        names = np.array(names)[체결]
        체결가 = 체결가[체결]
        주문수량 = np.array(주문수량)[체결]
        self._remove_assets(names, 체결가, 주문수량)

        거래대금 = np.sum(체결가 * 주문수량) * (100 - self._tax - self._fee - self._slippage) / 100
        return 거래대금

    def _get_current_price(self):
        names = list(self.keys())
        current_price = self._exchange.get_assets_info(codes=names, fields=["현재가", "대비"])

        return names, current_price

    def _apply_current_price(self):
        names, current_price = self._get_current_price()
        for i in range(len(names)):
            현재가 = current_price[i, 0]
            대비 = current_price[i, 1]

            if np.isnan(현재가):
                del self[names[i]]
                if self._출력:
                    print("\x1b[31m\"%s\"\x1b[0m" % (names[i] + '  상장폐지 !!! ###################################'))
            else:
                전일종가 = self[names[i]]["현재가"]
                등락률 = 현재가 / 전일종가

                if (등락률 > 1.35) or ((등락률 < 0.65)):
                    수정전일종가 = 현재가 - 대비
                    수정계수 = 수정전일종가 / 전일종가
                    if 수정계수 == 0:
                        print(names[i])
                    self[names[i]]["평단가"] = self[names[i]]["평단가"] * 수정계수
                    self[names[i]]["보유수량"] = int(self[names[i]]["보유수량"] / 수정계수)

                self[names[i]]["현재가"] = current_price[i, 0]

# Updater.py

In [2]:
import numpy as np
import pandas as pd


class Updater:
    def __init__(self, date, list_date):
        self._list_date = list_date

        self._date = date
        self._date_start = date

        self.state = True

        # 날짜가 업데이트 될 경우 내부 정보를 업데이트할 instance를 넣는다.
        # 해당 instance들은 "update_date" method를 갖고 있어야한다.
        # 해당 instance들은 "reset" method를 갖고 있어야한다.

        self.list_instance4update = list()
        self.update()

    def set_instance4update(self, instance):
        """
        날짜가 변할 경우 업데이트가 필요한 instance를 등록한다.
        :param instance: class, method로 "update_date"과 "reset"를 가져야한다.
        :return:
        """

        methods = dir(instance)
        if (('update_date' not in methods)
                or ('reset' not in methods)):
            raise Exception

        # 업데이트 순서를 고려하여 Account는 대응되는 Exchange가 업데이트 된 후 업데이트되어야 한다.
        if "type" in methods:
            if instance.type == "Account":
                if instance._exchange not in self.list_instance4update:
                    raise Exception

        self.list_instance4update.append(instance)
        instance.update_date(self._date)

    def update(self):
        self._date = self._date + pd.Timedelta(days=1)

        while self._date not in self._list_date:
            self._date = self._date + pd.Timedelta(days=1)

        for instance in self.list_instance4update:
            instance.update_date(self._date)

    def reset(self):
        self._date = self._date_start
        self.update()
        for instance in self.list_instance4update:
            instance.reset(self._date)


# Exchange.py

In [4]:
import numpy as np
from data.loader import *


class Exchange:
    def __init__(self):
        self.fields = ["시가", "현재가", "고가", "저가", "거래량(주)", "거래대금(원)", "대비", "시장구분"]

        self._date = None
        self._OCLHVVM = None

        self._DataAsset = None

    def buy(self, list_codes, 주문가격, 주문종류=None, 주문시간=None):
        OCLHVVM = self.get_assets_info(codes=list_codes)
        market_type = OCLHVVM[:, -1]
        OCLHVVM = OCLHVVM[:, :-2].astype("int64")

        주문가격 = cal_price_tick_unit(주문가격, market_type)

        if 주문종류 is None:
            주문종류 = ["limit" for x in np.arange(len(list_codes))]
        else:
            주문종류 = np.array(주문종류)

        if 주문시간 is None:
            주문시간 = ["장중" for x in np.arange(len(list_codes))]
        else:
            주문시간 = np.array(주문시간)

        주문시간 = np.array(주문시간)

        체결가 = np.zeros_like(list_codes, dtype="int") * np.nan  # 체결가가 nan인 경우 미체결, 숫자인 경우 체결가격

        # 체결조건 (시간순서로)
        cond1 = (OCLHVVM[:, 4] == 0) | (np.isnan(OCLHVVM[:, 4]))  # 미체결, 거래량이 0이거나 상장되지 않음
        cond2 = (주문시간 == "장전") & (OCLHVVM[:, 0] <= 주문가격)  # 체결, 장 시작과 동시에 체결
        cond3 = OCLHVVM[:, 2] <= 주문가격  # 체결, 장중 체결
        cond4 = OCLHVVM[:, 3] <= 주문가격  # 체결, 장중 체결
        cond5 = (주문종류 == "조건부지정가")  # 체결, 장 마감시 체결

        체결성공1 = ~cond1 & cond2  # 장시작과 동시에 시가에 체결
        체결성공2 = ~cond1 & ~cond2 & cond3  # 고가 < 주문가격, 고가에 장중 체결
        체결성공3 = ~cond1 & ~cond2 & ~cond3 & cond4  # 저가 < 주문가격, 주문가격에 장중 체결
        체결성공4 = ~cond1 & ~cond2 & ~cond3 & ~cond4 & cond5  # 장마감 동시호가에 체결

        체결가[체결성공1] = OCLHVVM[체결성공1, 0]  # 시가 체결
        체결가[체결성공2] = OCLHVVM[체결성공2, 2]  # 고가 체결
        체결가[체결성공3] = 주문가격[체결성공3]  # 시가 체결
        체결가[체결성공4] = OCLHVVM[체결성공4, 1]  # 종가 체결

        현재가 = OCLHVVM[:, 1]
        return 체결가, 현재가

    def sell(self, list_codes, 주문가격, 주문종류=None, 주문시간=None):
        OCLHVVM = self.get_assets_info(codes=list_codes)
        market_type = OCLHVVM[:, -1]
        OCLHVVM = OCLHVVM[:, :-2].astype("int64")

        주문가격 = cal_price_tick_unit(주문가격, market_type)

        if 주문종류 is None:
            주문종류 = ["limit" for x in np.arange(len(list_codes))]
        else:
            주문종류 = np.array(주문종류)

        if 주문시간 is None:
            주문시간 = ["장중" for x in np.arange(len(list_codes))]
        else:
            주문시간 = np.array(주문시간)

        주문시간 = np.array(주문시간)

        체결가 = np.zeros_like(list_codes, dtype="int") * np.nan  # 체결가가 nan인 경우 미체결, 숫자인 경우 체결가격

        # 체결조건 (시간순서로)
        cond1 = (OCLHVVM[:, 4] == 0) | (np.isnan(OCLHVVM[:, 4]))  # 미체결, 거래량이 0이거나 상장되지 않음
        cond2 = (주문시간 == "장전") & (OCLHVVM[:, 0] >= 주문가격)  # 체결, 장 시작과 동시에 체결
        cond3 = OCLHVVM[:, 3] >= 주문가격  # 체결, 장중 체결 : 저가 > 판매가
        cond4 = OCLHVVM[:, 2] >= 주문가격  # 체결, 장중 체결 : 고가 > 판매가
        cond5 = (주문종류 == "조건부지정가")  # 체결, 장 마감시 체결

        체결성공1 = ~cond1 & cond2  # 장시작과 동시에 시가에 체결
        체결성공2 = ~cond1 & ~cond2 & cond3  # 저가 > 주문가격, 저가에 장중 체결
        체결성공3 = ~cond1 & ~cond2 & ~cond3 & cond4  # 고가 > 주문가격, 주문가격에 장중 체결
        체결성공4 = ~cond1 & ~cond2 & ~cond3 & ~cond4 & cond5  # 장마감 동시호가에 체결

        체결가[체결성공1] = OCLHVVM[체결성공1, 0]  # 시가 체결
        체결가[체결성공2] = OCLHVVM[체결성공2, 3]  # 고가 체결
        체결가[체결성공3] = 주문가격[체결성공3]  # 시가 체결
        체결가[체결성공4] = OCLHVVM[체결성공4, 1]  # 종가 체결

        return 체결가

    def set_DataAsset(self, data_asset):
        self._DataAsset = data_asset

        self.codes = data_asset.codes

    def get_assets_info(self, codes=None, fields=None):
        array = self._OCLHVVM[:]

        if codes is not None:
            idx_codes = [self.codes.index(code) for code in codes]
            array = array[idx_codes, :]
        if fields is not None:
            idx_fields = [self.fields.index(field) for field in fields]
            array = array[:, idx_fields]

        return array

    def update_date(self, date):
        self._date = date
        self._get_OCLHVV()

    def reset(self, date):
        self.update_date(date)

    def _get_OCLHVV(self):
        self._OCLHVVM = self._DataAsset.get_info(self._date, num=1, fields=self.fields).reshape(-1, 8)


#     def get_assets_info(self, codes, fields=None):

#         if fields is None:
#             array = self._DataAsset.get_info(self._date, num=1, codes=codes, fields=self.fields).reshape(-1, 8)
#         else:
#             array = self._DataAsset.get_info(self._date, num=1, codes=codes, fields=fields).reshape(-1, 8)

#         return array

def cal_price_tick_unit(price, market_type):
    price = np.array(price)
    market_type = np.array(market_type).reshape(-1)
    tick_unit = np.zeros_like(price)

    kosdaq = (market_type == 1)  # type : 0: 코스피, 1: 코스닥
    tick_unit[kosdaq] = 100
    tick_unit[kosdaq & (price < 50000)] = 50
    tick_unit[kosdaq & (price < 10000)] = 10
    tick_unit[kosdaq & (price < 5000)] = 5
    tick_unit[kosdaq & (price < 1000)] = 1

    tick_unit[~kosdaq] = 1000
    tick_unit[~kosdaq & (price < 500000)] = 500
    tick_unit[~kosdaq & (price < 100000)] = 100
    tick_unit[~kosdaq & (price < 50000)] = 50
    tick_unit[~kosdaq & (price < 10000)] = 10
    tick_unit[~kosdaq & (price < 5000)] = 5
    tick_unit[~kosdaq & (price < 1000)] = 1

    price = ((price / tick_unit) * tick_unit).astype("i")
    return price


# DataStock.py

In [5]:
import numpy as np
import dask.array as da
import pandas as pd

from data.loader import *


def make_data(data_df, fields=None, dtype=None, dates=None):
    """
    sql에서 읽은 dataframe을 (날짜, 종목코드, 필드)로 구성된 3-dimensional array로 변환한다.
    :param data_df: pandas.dataframe, 종목코드와 날짜를 각각 table과 index로 갖는 pandas dataframe
    :param dates:
    :return: numpy.array, (list, list, list), 두번째 tuple은 각각 날짜, 종목코드, 필드의 리스트로 구성된다.
    """

    if dtype == "stock":
        fields = ["현재가", "시가", "고가", "저가", "대비", "거래량(주)", "거래대금(원)", "상장시가총액(원)", "시장구분"]
    else:
        fields = list(data_df["A005930"].columns)

    if dates is None:
        dates = data_df["A005930"].index  # A005930 : 삼성전자
    else:
        dates = dates

    codes = list(data_df.keys())

    list_data = list()

    for code in codes:
        dummy = data_df[code][fields].reindex(dates).fillna(np.nan)
        list_data.append(np.array(dummy).reshape(len(dates), 1, -1))

    dates = list(dates)
    array = np.concatenate(list_data, axis=1)

    if dtype == "stock":
        array[:, :, -1][np.where(array[:, :, -1] == "코스피")] = 0
        array[:, :, -1][np.where(array[:, :, -1] == "코스닥")] = 1
        array = array.astype('f')

    return array, (dates, codes, fields)


class DataStock:
    def __init__(self, array, axis, chunk=300):
        #         array, axis = make_data(data_df)

        self.dates, self.codes, self.fields = axis
        self.array = array

        self._chunk = chunk
        self._dates_chunk = []

        self._date = None
        self.df_date = None

    def get_info(self, date, num=1, codes=None, fields=None):
        """

        :param date: Pandas.Timestamp
        :param num: int, 반환할 과거 일수
        :param codes: list, 반환할 종목코드들의 리스트
        :param fields: list, 반환할 필드들의 리스트
        :return: numpy.array
        """
        idx_date = self._dates_chunk.index(date)
        array = self._array_chunk[max(0, idx_date - num + 1):idx_date + 1]

        if codes is not None:
            idx_codes = [self.codes.index(code) for code in codes]
            array = array[:, idx_codes, :]
        if fields is not None:
            idx_fields = [self.fields.index(field) for field in fields]
            array = array[:, :, idx_fields]

        if num == 1:
            array = array[0]

        return array.compute()

    def _make_chunk(self):
        idx_date = self.dates.index(self._date)
        self._dates_chunk = self.dates[max(0, idx_date - self._chunk):idx_date + self._chunk]
        self._array_chunk = self.array[max(0, idx_date - self._chunk):idx_date + self._chunk]

    def update_date(self, date):
        self._date = date
        if date not in self._dates_chunk:
            self._make_chunk()

    def reset(self, date):
        self.update_date(date)


In [6]:
data_stock_df = read_db("../data/상장종목검색.db", date_format="%Y/%m/%d", num_process=6)
data_value_df = read_db("../data/재무데이터(밸류).db", date_format="%Y/%m/%d", num_process=6)

array_stock, axis_stock = make_data(data_stock_df, dtype="stock")
array_value, axis_value = make_data(data_value_df)

In [7]:
array_stock = da.from_array(array_stock, chunks=(50, len(axis_stock[1]), len(axis_stock[2])))
data_stock = DataStock(array_stock, axis_stock, chunk=50)

array_value = da.from_array(array_value, chunks=(50, len(axis_stock[1]), len(axis_value[2])))
data_value = DataStock(array_value, axis_value, chunk=50)

In [8]:
import h5py

f = h5py.File("dummy3.hdf5", "a")
f.create_dataset("stock", data=array_stock)
f.create_dataset("value", data=array_value)

<HDF5 dataset "value": shape (4430, 2761, 36), type "<f8">

In [9]:
array_stock = da.from_array(f["stock"], chunks=(5, len(axis_stock[1]), len(axis_stock[2])))
data_stock = DataStock(array_stock, axis_stock, chunk=5)

array_value = da.from_array(f["value"], chunks=(5, len(axis_stock[1]), len(axis_value[2])))
data_value = DataStock(array_value, axis_value, chunk=5)

# del data_stock_df, data_value_df

# Agent.py

In [52]:
import numpy as np

class Agent:
    def __init__(self, initial_cash, 출력=True):
        self._출력 = 출력
        self._date = None

        self._initial_cash = initial_cash * 1
        self.cash = initial_cash
        self.total_balance = initial_cash

        self.accounts = dict()  # 여러 자산군에 대한 계좌

        self.report = {"수익률(%)": [], "누적수익률(%)": [], "총자산(원)": [],
                       "CAGR(%)": [], "일평균수익률(%)": [], "MDD": [], "최대수익률(%)": [], "날짜": []}

    def set_account(self, name, account):
        self.accounts[name] = account

    def buy(self, name_account, names, 주문가격, 주문수량, **kwds):
        주문가격 = np.array(주문가격)
        주문수량 = np.array(주문수량)

        cash_required = np.sum(주문가격 * 주문수량)
        if cash_required > self.cash:
            idx = np.sum(np.cumsum(주문가격 * 주문수량) < cash_required) - 1
            if idx == -1:
                return False

            names = names[:idx]
            주문가격 = 주문가격[:idx]
            주문수량 = 주문수량[:idx]

        cash_consumed = self.accounts[name_account].buy(names, 주문가격, 주문수량, **kwds)
        self.cash -= cash_consumed
        print(self.cash)

        return True

    def sell(self, name_account, names, 주문가격, 주문수량, **kwds):
        주문가격 = np.array(주문가격)
        주문수량 = np.array(주문수량)

        cash_earned = self.accounts[name_account].sell(names, 주문가격, 주문수량, **kwds)
        self.cash += cash_earned

    def update_date(self, date):
        self._date = date

        self.total_balance = 0
        for key in self.accounts.keys():
            self.accounts[key].update_from_agent(date)
            self.total_balance += self.accounts[key].get_total_balance()

        self.total_balance += self.cash
        self._update_report()

    def reset(self, date):
        self._date = date

        for key in self.accounts.keys():
            self.accounts[key].reset_from_agent(date)
        
        self.cash = self.initial_cash * 1
        self.total_balance = self.initial_cash * 1

        self.report = {"수익률(%)": [], "누적수익률(%)": [], "총자산(원)": [],
                       "CAGR(%)": [], "일평균수익률(%)": [], "MDD": [], "최대수익률(%)": [], "날짜": []}
        self._update_report()

    def _update_report(self):
        if not self.report["날짜"]:
            self.report = {"수익률(%)": [0.0], "누적수익률(%)": [0.0], "총자산(원)": [self._initial_cash],
                           "CAGR(%)": [0.0], "일평균수익률(%)": [0.0], "MDD": [0.0], "최대수익률(%)": [0], "날짜": [self._date]}
        else:
            당일수익률 = (self.total_balance - self.report["총자산(원)"][-1]) / self.report["총자산(원)"][-1] * 100
            누적수익률 = (self.total_balance / self._initial_cash - 1) * 100
            총자산 = self.total_balance
            경과 = (self._date - self.report["날짜"][0]).days + 1

            일평균수익률 = ((누적수익률 / 100 + 1) ** (1 / 경과) - 1) * 100
            CAGR = ((일평균수익률 / 100 + 1) ** 365 - 1) * 100

            최대수익률 = max(누적수익률, np.max(self.report["누적수익률(%)"]))
            DD = (누적수익률 - 최대수익률) / (최대수익률 + 100) * 100
            MDD = min(DD, np.min(self.report["MDD"]))

            레포트_당일 = {"수익률(%)": 당일수익률, "누적수익률(%)": 누적수익률, "총자산(원)": 총자산,
                      "CAGR(%)": CAGR, "일평균수익률(%)": 일평균수익률, "MDD": MDD, "최대수익률(%)": 최대수익률, "날짜": self._date}

            for field in self.report.keys():
                self.report[field].append(레포트_당일[field])

            if self._출력:
                print("장마감 : ", self._date, "\n\n",
                      "당일수익률(%) : ", 당일수익률, "\n",
                      "누적수익률(%) : ", 누적수익률, "\n",
                      "CAGR(%)", CAGR, "\n",
                      "MDD : ", MDD, "\n",
                      "총자산(원) : ", 총자산, "\n",
                      "--------------------------------------------------\n")

In [111]:
updater = Updater(pd.Timestamp(2002, 6, 17), data_stock.dates)

# 거래소 생성
exchange_stock = Exchange()
exchange_stock.set_DataAsset(data_stock)

# 주식 계좌 생성
stock_account = StockAccount(exchange_stock, 출력=True)

# 거래 에이전트 생성 및 주식 계좌 등록
agent = Agent(1e8, 출력=True)
agent.set_account("stock", stock_account)

# 날짜가 변할시 업데이트 요청
updater.set_instance4update(data_stock)
updater.set_instance4update(data_value)

updater.set_instance4update(exchange_stock)
updater.set_instance4update(agent)

In [112]:
%%time
columns = ["상장시가총액(원)", "지배주주순이익(원)(직전4분기)", "지배주주지분(원)",
   "현금흐름(원)(직전4분기)", "매출액(원)(직전4분기)"]


while updater._date != updater._list_date[-1]:
    print(updater._date)
    fin_stat = data_value.get_info(updater._date, num=2,
                                 fields=columns)
    
    df = pd.DataFrame(fin_stat[-2], index=data_value.codes, columns=columns)
    df = df[~np.isnan(df["상장시가총액(원)"])] # 상장종목 고려 
    df = df.sort_values(by=['상장시가총액(원)']).iloc[:int(len(df.index) * 0.3)] # 소형주

    # 종목선정
    df["PER"] = df["상장시가총액(원)"] / df["지배주주순이익(원)(직전4분기)"]
    df["PBR"] = df["상장시가총액(원)"] / df["지배주주지분(원)"]
    df["PCR"] = df["상장시가총액(원)"] / df["현금흐름(원)(직전4분기)"]
    df["PSR"] = df["상장시가총액(원)"] / df["매출액(원)(직전4분기)"]

    df = df[df["PER"] > 0]
    df = df[df["PBR"] > 0]
    df = df[df["PCR"] > 0]
    df = df[df["PSR"] > 0]

    df["Rank"] = (df["PER"].rank() + df["PBR"].rank() + df["PCR"].rank() + df["PSR"].rank()).rank()

    df = df[df["Rank"] < 51]

    매수종목 = np.sort(df.index)

    # 매도
    매도종목 = list(agent.accounts["stock"].keys())
    현재가 = data_stock.get_info(updater._date, codes=매도종목, fields=["현재가"]).reshape(-1)
    매도수량 = [agent.accounts["stock"][종목코드]["보유수량"] for 종목코드 in 매도종목]
    agent.sell("stock", 매도종목, 현재가, 매도수량)

    # 매수
    현재가 = data_stock.get_info(updater._date, codes=매수종목, fields=["현재가"]).reshape(-1).astype("f")
    거래가능 = ~np.isnan(현재가)
    매수수량 = (agent.cash / 50 / 현재가[거래가능]).astype("i")
    agent.buy("stock", 매수종목[거래가능], 현재가[거래가능], 매수수량)

    for i in range(21):
        if updater._date == updater._list_date[-1]:
            break
        updater.update()

2002-06-18 00:00:00
매수 실패 :  A000040 주문가 3145.0 주문수량 :  635
매수 체결 :  A000300 체결가 :  8480.0 주문수량 :  235
매수 체결 :  A000420 체결가 :  2660.0 주문수량 :  751
매수 실패 :  A000470 주문가 835.0 주문수량 :  2395
매수 체결 :  A000590 체결가 :  23200.0 주문수량 :  86
매수 체결 :  A001070 체결가 :  13200.0 주문수량 :  151
매수 체결 :  A001600 체결가 :  4250.0 주문수량 :  470
매수 체결 :  A001670 체결가 :  3510.0 주문수량 :  569
매수 체결 :  A001810 체결가 :  8920.0 주문수량 :  224
매수 체결 :  A001840 체결가 :  12600.0 주문수량 :  158
매수 체결 :  A001950 체결가 :  5630.0 주문수량 :  355
매수 체결 :  A002050 체결가 :  3600.0 주문수량 :  555
매수 체결 :  A002290 체결가 :  7160.0 주문수량 :  279
매수 체결 :  A002360 체결가 :  3410.0 주문수량 :  586
매수 체결 :  A003280 체결가 :  7800.0 주문수량 :  256
매수 체결 :  A003650 체결가 :  9370.0 주문수량 :  213
매수 체결 :  A003960 체결가 :  5100.0 주문수량 :  392
매수 체결 :  A004090 체결가 :  13500.0 주문수량 :  148
매수 체결 :  A004100 체결가 :  3700.0 주문수량 :  540
매수 체결 :  A004780 체결가 :  6350.0 주문수량 :  314
매수 체결 :  A004920 체결가 :  5370.0 주문수량 :  372
매수 체결 :  A004960 체결가 :  7110.0 주문수량 :  281
매수 체결 :  A005030 체결가 :  10300.0 주문수량 

c:\users\wkwek\appdata\local\programs\python\python36\lib\site-packages\ipykernel_launcher.py:78: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison


장마감 :  2002-07-03 00:00:00 

 당일수익률(%) :  0.06711519241101917 
 누적수익률(%) :  -7.37314 
 CAGR(%) -82.57445024912417 
 MDD :  -13.073515000000002 
 총자산(원) :  92626860.0 
 --------------------------------------------------

장마감 :  2002-07-04 00:00:00 

 당일수익률(%) :  2.094862116668966 
 누적수익률(%) :  -5.432734999999999 
 CAGR(%) -69.8601333661265 
 MDD :  -13.073515000000002 
 총자산(원) :  94567265.0 
 --------------------------------------------------

장마감 :  2002-07-05 00:00:00 

 당일수익률(%) :  0.787762023148285 
 누적수익률(%) :  -4.687770000000002 
 CAGR(%) -62.22707918532786 
 MDD :  -13.073515000000002 
 총자산(원) :  95312230.0 
 --------------------------------------------------

장마감 :  2002-07-08 00:00:00 

 당일수익률(%) :  0.45453768105100467 
 누적수익률(%) :  -4.254539999999995 
 CAGR(%) -53.03047590717544 
 MDD :  -13.073515000000002 
 총자산(원) :  95745460.0 
 --------------------------------------------------

장마감 :  2002-07-09 00:00:00 

 당일수익률(%) :  0.6803977964072657 
 누적수익률(%) :  -3.6030899999999977 

 누적수익률(%) :  -10.253538297499997 
 CAGR(%) -63.667904595617685 
 MDD :  -13.073515000000002 
 총자산(원) :  89746461.7025 
 --------------------------------------------------

장마감 :  2002-07-29 00:00:00 

 당일수익률(%) :  -0.20944001182250951 
 누적수익률(%) :  -10.441503297499999 
 CAGR(%) -61.648244964462606 
 MDD :  -13.073515000000002 
 총자산(원) :  89558496.7025 
 --------------------------------------------------

장마감 :  2002-07-30 00:00:00 

 당일수익률(%) :  2.5638550048773916 
 누적수익률(%) :  -8.145353297500002 
 CAGR(%) -51.382942401717614 
 MDD :  -13.073515000000002 
 총자산(원) :  91854646.7025 
 --------------------------------------------------

장마감 :  2002-07-31 00:00:00 

 당일수익률(%) :  -0.6745004441709289 
 누적수익률(%) :  -8.764913297499998 
 CAGR(%) -53.27764101940195 
 MDD :  -13.073515000000002 
 총자산(원) :  91235086.7025 
 --------------------------------------------------

장마감 :  2002-08-01 00:00:00 

 당일수익률(%) :  -0.6767791014584311 
 누적수익률(%) :  -9.382373297500003 
 CAGR(%) -55.027469658819406 


매수 실패 :  A000040 주문가 3145.0 주문수량 :  611
매수 체결 :  A000300 체결가 :  7350.0 주문수량 :  261
매수 체결 :  A000420 체결가 :  2425.0 주문수량 :  792
매수 체결 :  A000590 체결가 :  23950.0 주문수량 :  80
매수 체결 :  A001070 체결가 :  14200.0 주문수량 :  135
매수 체결 :  A001600 체결가 :  3250.0 주문수량 :  591
매수 체결 :  A001670 체결가 :  3800.0 주문수량 :  505
매수 체결 :  A001950 체결가 :  4665.0 주문수량 :  412
매수 체결 :  A002050 체결가 :  2245.0 주문수량 :  856
매수 체결 :  A002290 체결가 :  6030.0 주문수량 :  318
매수 체결 :  A002360 체결가 :  3600.0 주문수량 :  533
매수 체결 :  A002700 체결가 :  2460.0 주문수량 :  781
매수 체결 :  A003280 체결가 :  11350.0 주문수량 :  169
매수 체결 :  A003310 체결가 :  3700.0 주문수량 :  519
매수 체결 :  A003960 체결가 :  3080.0 주문수량 :  624
매수 체결 :  A004090 체결가 :  13300.0 주문수량 :  144
매수 체결 :  A004100 체결가 :  3210.0 주문수량 :  598
매수 체결 :  A004780 체결가 :  6480.0 주문수량 :  296
매수 체결 :  A004920 체결가 :  4700.0 주문수량 :  408
매수 체결 :  A004960 체결가 :  7700.0 주문수량 :  249
매수 체결 :  A005030 체결가 :  8650.0 주문수량 :  222
매수 체결 :  A005320 체결가 :  6220.0 주문수량 :  309
매수 체결 :  A005820 체결가 :  5550.0 주문수량 :  346
매수 체결 :  A0

...

매수 체결 :  A212560 체결가 :  6160.0 주문수량 :  30839
매수 체결 :  A214330 체결가 :  6180.0 주문수량 :  30740
매수 체결 :  A900080 체결가 :  722.0 주문수량 :  263122
137211.21090698242
장마감 :  2018-05-31 00:00:00 

 당일수익률(%) :  0.9414791002699686 
 누적수익률(%) :  9697.628596210907 
 CAGR(%) 33.2670231909552 
 MDD :  -59.02969290815421 
 총자산(원) :  9797628596.210907 
 --------------------------------------------------

장마감 :  2018-06-01 00:00:00 

 당일수익률(%) :  -0.29425275429555997 
 누적수익률(%) :  9668.798804210906 
 CAGR(%) 33.235864338076816 
 MDD :  -59.02969290815421 
 총자산(원) :  9768798804.210907 
 --------------------------------------------------

장마감 :  2018-06-04 00:00:00 

 당일수익률(%) :  1.1700267176219379 
 누적수익률(%) :  9783.096360210906 
 CAGR(%) 33.3132315657227 
 MDD :  -59.02969290815421 
 총자산(원) :  9883096360.210907 
 --------------------------------------------------

장마감 :  2018-06-05 00:00:00 

 당일수익률(%) :  -0.26639947684814586 
 누적수익률(%) :  9756.767843210908 
 CAGR(%) 33.28440531072354 
 MDD :  -59.0296929081

매도 체결 :  A073110 체결가 :  6890.0 주문수량 :  23581  / 수익률(%) :  -4.355668523676869 , 수익금(원) :  -7374651.19699998
매도 체결 :  A079660 체결가 :  11650.0 주문수량 :  12826  / 수익률(%) :  -12.03367424242424 , 수익금(원) :  -20373395.569999997
매도 체결 :  A081580 체결가 :  3395.0 주문수량 :  49580  / 수익률(%) :  -0.913718887262068 , 수익금(원) :  -1547069.5299999812
매도 체결 :  A083450 체결가 :  6930.0 주문수량 :  23067  / 수익률(%) :  -5.897397820163496 , 수익금(원) :  -9984989.223000014
매도 체결 :  A085670 체결가 :  3705.0 주문수량 :  53327  / 수익률(%) :  16.30782677165355 , 수익금(원) :  27611307.434500013
매도 체결 :  A088790 체결가 :  5880.0 주문수량 :  29861  / 수익률(%) :  3.3614814814814737 , 수익금(원) :  5691387.155999986
매도 체결 :  A091340 체결가 :  3605.0 주문수량 :  50391  / 수익률(%) :  6.937604166666672 , 수익금(원) :  11746318.468500009
매도 체결 :  A092300 체결가 :  3590.0 주문수량 :  47560  / 수익률(%) :  0.5099157303370728 , 수익금(원) :  863356.6799999902
매도 체결 :  A123700 체결가 :  3745.0 주문수량 :  44498  / 수익률(%) :  -1.9016688567674067 , 수익금(원) :  -3219808.532999992
매도 체결 :  A155660 체결가 :  4590.

매도 체결 :  A019180 체결가 :  2400.0 주문수량 :  74757  / 수익률(%) :  4.915789473684208 , 수익금(원) :  8378764.559999995
매도 체결 :  A019540 체결가 :  3230.0 주문수량 :  52688  / 수익률(%) :  -0.48404945904173435 , 수익금(원) :  -825041.3920000056
매도 체결 :  A023150 체결가 :  5340.0 주문수량 :  32281  / 수익률(%) :  0.802613636363648 , 수익금(원) :  1368004.2180000197
매도 체결 :  A024830 체결가 :  6740.0 주문수량 :  25747  / 수익률(%) :  1.4767069486404942 , 수익금(원) :  2516975.2260000184
매도 체결 :  A024900 체결가 :  1440.0 주문수량 :  114011  / 수익률(%) :  -3.996789297658875 , 수익금(원) :  -6812385.27200002
매도 체결 :  A025530 체결가 :  3900.0 주문수량 :  43873  / 수익률(%) :  0.05482625482625764 , 수익금(원) :  93449.4900000048
매도 체결 :  A025820 체결가 :  1785.0 주문수량 :  89006  / 수익률(%) :  -7.0961096605744105 , 수익금(원) :  -12095069.842999998
매도 체결 :  A025880 체결가 :  2510.0 주문수량 :  76951  / 수익률(%) :  12.944334085778786 , 수익금(원) :  22063159.867000006
매도 체결 :  A031310 체결가 :  5300.0 주문수량 :  44043  / 수익률(%) :  36.49896640826874 , 수익금(원) :  62211177.93000001
매도 체결 :  A036710 체결가 :  2010.0

장마감 :  2018-10-04 00:00:00 

 당일수익률(%) :  -0.2210260245373655 
 누적수익률(%) :  9074.479850973108 
 CAGR(%) 31.92678529287265 
 MDD :  -59.02969290815421 
 총자산(원) :  9174479850.973108 
 --------------------------------------------------

2018-10-04 00:00:00
매도 실패 :  A900090 주문가 :  773.0 주문수량 :  229304
매도 체결 :  A000910 체결가 :  3995.0 주문수량 :  43012  / 수익률(%) :  -2.763943833943825 , 수익금(원) :  -4868248.701999985
매도 체결 :  A001810 체결가 :  2910.0 주문수량 :  58712  / 수익률(%) :  -3.3201000000000023 , 수익금(원) :  -5847891.336000004
매도 체결 :  A002220 체결가 :  22000.0 주문수량 :  7543  / 수익률(%) :  -6.092505353319051 , 수익금(원) :  -10730671.79999999
매도 체결 :  A002710 체결가 :  2010.0 주문수량 :  82307  / 수익률(%) :  -6.3847196261682155 , 수익금(원) :  -11245852.330999985
매도 체결 :  A006200 체결가 :  860.0 주문수량 :  194626  / 수익률(%) :  -5.285966850828738 , 수익금(원) :  -9310518.588000016
매도 체결 :  A007280 체결가 :  4100.0 주문수량 :  60528  / 수익률(%) :  40.42852233676975 , 수익금(원) :  71209376.15999998
매도 체결 :  A008830 체결가 :  28600.0 주문수량 :  6621  / 수익률(

 MDD :  -59.02969290815421 
 총자산(원) :  8752416062.670809 
 --------------------------------------------------

장마감 :  2018-10-22 00:00:00 

 당일수익률(%) :  0.004504596184378498 
 누적수익률(%) :  8652.810323670808 
 CAGR(%) 31.438055673139242 
 MDD :  -59.02969290815421 
 총자산(원) :  8752810323.670809 
 --------------------------------------------------

장마감 :  2018-10-23 00:00:00 

 당일수익률(%) :  -1.7907466311261857 
 누적수익률(%) :  8496.069667670808 
 CAGR(%) 31.286966310652065 
 MDD :  -59.02969290815421 
 총자산(원) :  8596069667.670809 
 --------------------------------------------------

장마감 :  2018-10-24 00:00:00 

 당일수익률(%) :  -1.3603739560161767 
 누적수익률(%) :  8379.13097467081 
 CAGR(%) 31.171145888965192 
 MDD :  -59.02969290815421 
 총자산(원) :  8479130974.670809 
 --------------------------------------------------

장마감 :  2018-10-25 00:00:00 

 당일수익률(%) :  -2.25783043771691 
 누적수익률(%) :  8187.686574670809 
 CAGR(%) 30.98230080351092 
 MDD :  -59.02969290815421 
 총자산(원) :  8287686574.670809 
 ----

In [24]:
agent.report

{'수익률(%)': [0.0],
 '누적수익률(%)': [0.0],
 '총자산(원)': [100000000.0],
 'CAGR(%)': [0.0],
 '일평균수익률(%)': [0.0],
 'MDD': [0.0],
 '최대수익률(%)': [0],
 '날짜': [Timestamp('2002-06-17 00:00:00')]}

In [13]:
agent.report

{'수익률(%)': [0.0,
  0.01666,
  -4.278872139901493,
  0.03989572482760728,
  0.47964364913823787,
  -2.2241261333625673,
  -1.4762535487017436,
  -6.988054607508533,
  2.216436840888767,
  5.185568212148949,
  1.598644612386551,
  -0.0851942627192741,
  1.182021704940699,
  1.2240664063822966,
  -0.3596129055617864,
  1.5853263383191516,
  1.5867686868913087,
  0.1549301053899039,
  0.3635434452424239,
  -1.2826158887966614,
  -1.5868178066007461,
  0.031455210832825396,
  -1.1637815194271035,
  -4.293504897478475,
  1.8826960800986596,
  -2.1166720315274836,
  1.4768411449230574,
  -0.9305081703636692,
  -0.4973741604493308,
  2.1948551769368887,
  -0.2873849596640898,
  -0.7686724747197808,
  0.12259178661549466,
  -1.2442715092836159,
  -0.7346065265126577,
  1.1272714259238295,
  1.1141385842331237,
  1.1708989107780412,
  -0.12825066750472097,
  1.2576016626055972,
  0.15329564016109642,
  0.7523166238754747,
  -0.05604212922273448,
  1.346585834479168,
  0.6560468660156925,
  1.010

'c:\\users\\wkwek\\appdata\\local\\programs\\python\\python36\\lib\\site-packages\\ipykernel_launcher.py'